# Ernesto Investing AI — Notebook 01: Ingesta de Datos Reales

Descarga datos OHLCV reales de los 5 tickers del proyecto usando `yfinance`,
calcula indicadores tecnicos con la libreria `ta`, y guarda todo en la
coleccion `precios_ohlcv` de MongoDB Atlas.

**Antes de ejecutar:** configura el Secret `MONGO_URI` en Colab
(icono de llave en la barra lateral izquierda) con tu cadena de conexion
de MongoDB Atlas.

## 1. Instalacion de librerias

In [ ]:
!pip install yfinance ta "pymongo[srv]" --quiet

## 2. Importaciones

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import ta
from pymongo import MongoClient, UpdateOne
from datetime import datetime, timezone
from google.colab import userdata

## 3. Configuracion

Los 5 tickers son fijos para todo el proyecto (frontend, backend y API
deben usar exactamente estos mismos strings).

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
PERIODO = "1y"

MONGO_URI = userdata.get("MONGO_URI")
MONGO_DB_NAME = "ernesto_investing_ai"

## 4. Conexion a MongoDB Atlas

In [ ]:
cliente = MongoClient(MONGO_URI)
db = cliente[MONGO_DB_NAME]
coleccion = db["precios_ohlcv"]

# Indice unico ticker+fecha (idempotente: no falla si ya existe)
coleccion.create_index([("ticker", 1), ("fecha", 1)], unique=True)

print("Conexion exitosa a MongoDB Atlas. Base de datos:", MONGO_DB_NAME)

## 5. Descarga de datos OHLCV

**Nota tecnica importante:** yfinance (version 2.x) devuelve un
`MultiIndex` en las columnas incluso al descargar un solo ticker.
Si no se corrige, el codigo posterior falla porque las columnas no
se llaman `Open`, `High`, etc. sino tuplas como `('Close', 'BVN')`.

In [ ]:
def descargar_ticker(ticker: str, periodo: str = PERIODO) -> pd.DataFrame:
    """
    Descarga datos OHLCV reales de un ticker desde Yahoo Finance.
    Corrige el MultiIndex de columnas que yfinance 2.x devuelve
    incluso para un solo ticker.
    """
    df = yf.download(ticker, period=periodo, auto_adjust=False, progress=False)

    if df.empty:
        raise ValueError(f"yfinance no devolvio datos para el ticker {ticker}")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df["ticker"] = ticker
    return df

## 6. Calculo de indicadores tecnicos

In [ ]:
def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega columnas de indicadores tecnicos (SMA, EMA, RSI, MACD,
    Bandas de Bollinger) usando la libreria ta. Requiere al menos
    50 filas de historia para que SMA-50 no quede completamente en NaN.
    """
    df = df.copy()

    df["sma_20"] = ta.trend.sma_indicator(df["Close"], window=20)
    df["sma_50"] = ta.trend.sma_indicator(df["Close"], window=50)
    df["ema_12"] = ta.trend.ema_indicator(df["Close"], window=12)
    df["ema_26"] = ta.trend.ema_indicator(df["Close"], window=26)
    df["rsi_14"] = ta.momentum.rsi(df["Close"], window=14)

    macd = ta.trend.MACD(df["Close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()
    df["macd_hist"] = macd.macd_diff()

    bollinger = ta.volatility.BollingerBands(df["Close"])
    df["bb_upper"] = bollinger.bollinger_hband()
    df["bb_lower"] = bollinger.bollinger_lband()

    return df

## 7. Transformacion al esquema de MongoDB

Construye los documentos exactamente en el formato de la coleccion
`precios_ohlcv` definido en el contrato de datos del proyecto.
Los valores NaN (dias iniciales sin suficiente historia para el
indicador) se convierten a `None` para que el documento sea JSON
valido y el frontend pueda manejarlos como "dato no disponible".

In [ ]:
def limpiar_nan(valor):
    """Convierte NaN/NaT de pandas a None para que Mongo/JSON lo acepten."""
    if pd.isna(valor):
        return None
    return float(valor) if isinstance(valor, (np.floating, np.integer)) else valor


def construir_documentos(df: pd.DataFrame, ticker: str) -> list[dict]:
    """Convierte cada fila del DataFrame en un documento para precios_ohlcv."""
    documentos = []
    ahora = datetime.now(timezone.utc)

    for _, fila in df.iterrows():
        documentos.append({
            "ticker": ticker,
            "fecha": fila["Date"].strftime("%Y-%m-%d"),
            "open": limpiar_nan(fila["Open"]),
            "high": limpiar_nan(fila["High"]),
            "low": limpiar_nan(fila["Low"]),
            "close": limpiar_nan(fila["Close"]),
            "volume": limpiar_nan(fila["Volume"]),
            "adj_close": limpiar_nan(fila.get("Adj Close", fila["Close"])),
            "indicadores": {
                "sma_20": limpiar_nan(fila["sma_20"]),
                "sma_50": limpiar_nan(fila["sma_50"]),
                "ema_12": limpiar_nan(fila["ema_12"]),
                "ema_26": limpiar_nan(fila["ema_26"]),
                "rsi_14": limpiar_nan(fila["rsi_14"]),
                "macd": limpiar_nan(fila["macd"]),
                "macd_signal": limpiar_nan(fila["macd_signal"]),
                "macd_hist": limpiar_nan(fila["macd_hist"]),
                "bb_upper": limpiar_nan(fila["bb_upper"]),
                "bb_lower": limpiar_nan(fila["bb_lower"]),
            },
            "actualizado_en": ahora,
        })
    return documentos

## 8. Ejecucion: descarga, calculo e insercion en MongoDB

Se usa `UpdateOne` con `upsert=True` en vez de `insert_many` para que
el notebook sea **idempotente**: se puede volver a ejecutar sin generar
duplicados ni errores de clave unica (ticker+fecha ya existente se
actualiza en vez de fallar).

In [ ]:
resumen = {}

for ticker in TICKERS:
    print(f"Procesando {ticker}...")
    try:
        df = descargar_ticker(ticker)
        df = calcular_indicadores(df)
        documentos = construir_documentos(df, ticker)

        operaciones = [
            UpdateOne(
                {"ticker": doc["ticker"], "fecha": doc["fecha"]},
                {"$set": doc},
                upsert=True,
            )
            for doc in documentos
        ]
        resultado = coleccion.bulk_write(operaciones)
        resumen[ticker] = resultado.upserted_count + resultado.modified_count
        print(f"  {ticker}: {len(documentos)} dias procesados, "
              f"{resultado.upserted_count} nuevos, {resultado.modified_count} actualizados")

    except Exception as error:
        print(f"  ERROR con {ticker}: {error}")
        resumen[ticker] = f"ERROR: {error}"

print()
print("Resumen final:", resumen)

## 9. Verificacion final

In [ ]:
print("Total de documentos en precios_ohlcv:", coleccion.count_documents({}))
print()

for ticker in TICKERS:
    ultimo = coleccion.find_one({"ticker": ticker}, sort=[("fecha", -1)])
    if ultimo:
        print(f"{ticker}: ultimo dato = {ultimo['fecha']}, close = {ultimo['close']:.2f}, "
              f"rsi_14 = {ultimo['indicadores']['rsi_14']}")
    else:
        print(f"{ticker}: SIN DATOS")